In [1]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages.utils import trim_messages,count_tokens_approximately # <--- Trim messages lib from the langchain.

In [3]:
model = ChatOpenAI()
MAX_TOKENs = 150

## **`How its work`**
- we did not delete the message of previous conversations. 
- we just truncate and push to the llm

In [4]:
def call_model(state: MessagesState):
    
    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",                      
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENs
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response]}

In [6]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [7]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Al Amin. i just testing short-term-memory via langchian"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 22
Hi, my name is Al Amin. i just testing short-term-memory via langchian


"Hi Al Amin! It's great that you're testing your short-term memory with the Langchian method. How is the test going so far?"

In [8]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Going very well, i set as max_token as 150 for testing as its work or not."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 81
Hi, my name is Al Amin. i just testing short-term-memory via langchian
Hi Al Amin! It's great that you're testing your short-term memory with the Langchian method. How is the test going so far?
Going very well, i set as max_token as 150 for testing as its work or not.


"That's a good approach to setting a maximum token limit for testing. It's important to find the right balance to challenge your memory without overwhelming it. Keep up the good work and let me know if you have any questions or need assistance with anything!"

In [9]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 142
Hi Al Amin! It's great that you're testing your short-term memory with the Langchian method. How is the test going so far?
Going very well, i set as max_token as 150 for testing as its work or not.
That's a good approach to setting a maximum token limit for testing. It's important to find the right balance to challenge your memory without overwhelming it. Keep up the good work and let me know if you have any questions or need assistance with anything!
Can you explain short term memory?


'Short-term memory refers to the system in our brain that is responsible for temporarily storing and managing information that is actively being used or processed. It is a limited-capacity system that can hold a small amount of information for a short period of time, usually up to around 20-30 seconds.\n\nShort-term memory is crucial for everyday tasks such as remembering a phone number long enough to dial it or recalling a list of items while shopping. It is often contrasted with long-term memory, which involves the storage of information over a longer period of time.\n\nShort-term memory is believed to involve processes such as encoding, storing, maintaining, and retrieving information. It is an essential component of cognitive functioning and plays a key role in various cognitive tasks, such as problem-solving, decision-making, and learning.\n\nFactors such as attention, rehearsal, and mnemonic strategies can influence the effectiveness of short-term memory. Additionally, the capaci

## **See the messages trimming is working**
- my current token is 142 + last_msg ~= 180+
- so its cut the last messages before pass the `stm`

In [10]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 8
What is my name?


"I'm sorry, I do not have that information."

## **See all messages are store in the state**

In [11]:
for item in graph.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Al Amin. i just testing short-term-memory via langchian
------------------------------------------------------------------------------------------------------------------------
Hi Al Amin! It's great that you're testing your short-term memory with the Langchian method. How is the test going so far?
------------------------------------------------------------------------------------------------------------------------
Going very well, i set as max_token as 150 for testing as its work or not.
------------------------------------------------------------------------------------------------------------------------
That's a good approach to setting a maximum token limit for testing. It's important to find the right balance to challenge your memory without overwhelming it. Keep up the good work and let me know if you have any questions or need assistance with anything!
--------------------------------------------------------------------------------------------------------------